In [1]:
import subprocess
subprocess.run(["pip", "install", "delta-spark"], check=True)

CompletedProcess(args=['pip', 'install', 'delta-spark'], returncode=0)

In [2]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("delta-concepts") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://titanic-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [8]:
dt = DeltaTable.forPath(spark, "s3a://titanic/lakehouse/silver/titanic")
dt.history().show(truncate=False)

+-------+-------------------+------+--------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------

In [5]:
novos_dados = spark.createDataFrame([
    (1, 0, 3, "male", 22.0, 1, 0, 7.25, "S"),
    (2, 1, 1, "female", 38.0, 1, 0, 71.28, "C"),
    (999, 1, 2, "male", 30.0, 0, 0, 13.0, "S"),  # passageiro novo
], ["PassengerId", "Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"])

In [6]:
dt.alias("silver") \
    .merge(novos_dados.alias("novo"), "silver.PassengerId = novo.PassengerId") \
    .whenMatchedUpdate(set={
        "Survived": "novo.Survived",
        "Age": "novo.Age"
    }) \
    .whenNotMatchedInsert(values={
        "PassengerId": "novo.PassengerId",
        "Survived": "novo.Survived",
        "Pclass": "novo.Pclass",
        "Sex": "novo.Sex",
        "Age": "novo.Age",
        "SibSp": "novo.SibSp",
        "Parch": "novo.Parch",
        "Fare": "novo.Fare",
        "Embarked": "novo.Embarked"
    }) \
    .execute()

print("MERGE executado")

MERGE executado


In [7]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
dt.vacuum(0)

DataFrame[]

In [ ]:
# Verificar historico de merge adicionado

In [9]:
dt.history().show(truncate=False)

+-------+-------------------+------+--------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------

In [ ]:
# Buscando o novo passageiro(PassengerId = 999) adicionado no arquivo silver

In [14]:
df = spark.read.format("delta") \
    .load("s3a://titanic/lakehouse/silver/titanic")

df.createOrReplaceTempView("df_analise")

df_analise = spark.sql("""
    SELECT * FROM df_analise WHERE PassengerId = 999
""")

df_analise.show()